# Setup path

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent   # .../UAV_nonGPS (repo root)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/home/nguyenduytan/UAV_nonGPS


# Imports

In [2]:
import os
import torch
import pandas as pd

from UAV_nonGPS.model import load_model
from UAV_nonGPS.dataset import process_uav
from UAV_nonGPS.satellite.preprocess import (
    process_satellite_as_drone,
    complete_segmentation_demo_uav_with_rotation,
)
from UAV_nonGPS.matching.features import (
    load_satellite_bounds,
    FeatureExtractor,
    DistanceMetric,
    Matcher,
    MatchVisualizer,
)
from UAV_nonGPS.matching.pipeline import LocalizationPipeline


# Config paths

In [3]:
DATA_ROOT = "/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset"
FLIGHT_ID = "03"

MODEL_PATH = "/home/nguyenduytan/UAV_nonGPS/best_model.pth"

FLIGHT_CSV = os.path.join(DATA_ROOT, FLIGHT_ID, f"{FLIGHT_ID}.csv")
UAV_IMG_PATH = os.path.join(DATA_ROOT, FLIGHT_ID, "drone", "03_0001.JPG")
SAT_IMG_PATH = os.path.join(DATA_ROOT, FLIGHT_ID, f"satellite{FLIGHT_ID}.tif")

OUT_DIR = os.path.join(DATA_ROOT, FLIGHT_ID, "output")
os.makedirs(OUT_DIR, exist_ok=True)

SAT_CSV_PATH = os.path.join(OUT_DIR, f"satellite{FLIGHT_ID}.csv") 
print(FLIGHT_CSV, UAV_IMG_PATH, SAT_IMG_PATH, SAT_CSV_PATH, sep="\n")

/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/03.csv
/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/drone/03_0001.JPG
/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/satellite03.tif
/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/output/satellite03.csv


# Load model

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model(
    model_path=MODEL_PATH,
    device=device,
    num_classes=2,
    pretrained=False,   
).to(device).eval()

print("Device:", device)

/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS/model.py:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device)


Device: cuda


# Ekeland's angels feature extraction on Satellite (only one time)

In [5]:
# import os
# if os.path.exists(SAT_CSV_PATH):
#     os.remove(SAT_CSV_PATH)

# process_satellite_as_drone(
#     tif_path=SAT_IMG_PATH,
#     output_dir=OUT_DIR,
#     model=model,
#     device=device,
#     scale=1.0,  # backward-compatible
#     scales=[0.75, 1.0, 1.25, 1.5],
#     scale_weights=[0.15, 0.40, 0.30, 0.15],
#     dynamic_threshold=True,
#     dynamic_quantile=0.65,
#     min_seg_threshold=0.30,
#     max_seg_threshold=0.80,
#     use_uncertainty_filter=True,
#     uncertainty_ratio=0.60,
#     use_cascade=True,
#     contour_method="marching_squares",
#     dynamic_large_filter=True,
#     dynamic_large_quantile=0.995,
#     fallback_max_size_ratio=0.90,
#     enable_otsu_cleanup=True,                                       
#     morph_open_kernel=3,
#     morph_close_kernel=5,
#     min_component_area=40,
#     enable_dual_path_fusion=True,
#     iou_dedup_threshold=0.85,
#     max_search_radius=120.0,
# )

# print("Satellite CSV path:", SAT_CSV_PATH)

In [6]:
# from UAV_nonGPS.satellite.preprocess import segment_and_save_satellite_mask

# segment_and_save_satellite_mask(
#     tif_path=SAT_IMG_PATH,
#     output_dir=OUT_DIR,
#     model=model,
#     device=device,
#     scale=1.0,
#     scales=[0.75],
#     scale_weights=None,
#     patch_size=500,
#     overlap=100,
#     dynamic_threshold=True,
#     dynamic_quantile=0.65,
#     min_seg_threshold=0.30,
#     max_seg_threshold=0.80,
#     use_uncertainty_filter=True,
#     uncertainty_ratio=0.60,
#     enable_otsu_cleanup=True,
#     morph_open_kernel=3,
#     morph_close_kernel=5,
#     min_component_area=40,
#     save_soft_mask_npy=False
# )

# Ekeland's angels feature extraction on UAV image

In [ ]:
UAV_CSV_ROTATED = complete_segmentation_demo_uav_with_rotation(
    model=model,
    device=device,
    uav_img_path=UAV_IMG_PATH,
    uav_csv_path=FLIGHT_CSV,
    method="marching_squares",
    max_search_radius=100.0,
)
print("UAV CSV:", UAV_CSV_ROTATED)

# Matching + visualize

In [8]:
# df_flight = pd.read_csv(FLIGHT_CSV, names=["num","filename","date","lat","lon","height","Omega","Kappa","Phi1","Phi2"], header=None)
# row = df_flight[df_flight["filename"].astype(str).str.strip() == os.path.basename(UAV_IMG_PATH)].iloc[0]
# gt_lat, gt_lon = float(row["lat"]), float(row["lon"])

# sat_bounds = load_satellite_bounds(
#     satellite_filename=os.path.basename(SAT_IMG_PATH),
#     csv_path=os.path.join(DATA_ROOT, "satellite_coordinates_range.csv"),
# )

# uav_base = os.path.splitext(os.path.basename(UAV_IMG_PATH))[0]
# UAV_ROTATED_IMG_PATH = os.path.join(OUT_DIR, f"{uav_base}_rotated.jpg")
# _, img_rotated, _ = process_uav(
#     img_path=UAV_IMG_PATH,     
#     csv_path=FLIGHT_CSV,
# )
# img_rotated.save(UAV_ROTATED_IMG_PATH, quality=95)
# print("UAV image:", UAV_IMG_PATH)
# print("Rotated image used:", UAV_ROTATED_IMG_PATH)

# # Khoi tao OOP pipeline (dung cho chay end-to-end neu can)
# localization_pipeline = LocalizationPipeline(model=model, device=device)

# # OOP matching: giu nguyen logic feature + metric nhu cell cu
# df_uav = pd.read_csv(UAV_CSV_ROTATED)
# df_sat = pd.read_csv(SAT_CSV_PATH)

# extractor = FeatureExtractor(
#     include_shape_features=False,
#     include_side_ratios=True,
#     include_radius_ratio=True,
#     include_elongation=True,
# )
# metric = DistanceMetric(metric="l1", vm_kappa=4.0, angle_feature_dims=5, linear_weight=0.0)
# matcher = Matcher(extractor=extractor, distance_metric=metric)
# results = matcher.match_dataframe(df_uav=df_uav, df_sat=df_sat, top_k=100)

# print(f"UAV triangles: {len(df_uav)} | SAT triangles: {len(df_sat)}")
# print("Top 100 matches:")
# for i, m in enumerate(results[:100]):
#     print(f"  Top {i+1}: Score={m['score']:.3f} | UAV#{m['uav_idx']} -> SAT#{m['sat_idx']}")

# visualizer = MatchVisualizer()
# visualizer.render(
#     uav_img_path=UAV_ROTATED_IMG_PATH,
#     sat_img_path=SAT_IMG_PATH,
#     matches=results,
#     uav_is_rotated=True,
#     gt_lat=gt_lat,
#     gt_lon=gt_lon,
#     sat_bounds=sat_bounds,
# )


In [9]:
# DATA_ROOT = r"D:\1-REFERENCES\12-IMACS\UAV_nonGPS\UAV_nonGPS_dataset"
# FLIGHT_ID = "01"

# MODEL_PATH = r"D:\1-REFERENCES\12-IMACS\UAV_nonGPS\best_model.pth"

FLIGHT_CSV = os.path.join(DATA_ROOT, FLIGHT_ID, f"{FLIGHT_ID}.csv")
UAV_IMG_PATH = os.path.join(DATA_ROOT, FLIGHT_ID, "drone", "03_0001.JPG")
SAT_IMG_PATH = os.path.join(DATA_ROOT, FLIGHT_ID, f"satellite{FLIGHT_ID}.tif")

OUT_DIR = os.path.join(DATA_ROOT, FLIGHT_ID, "output")
os.makedirs(OUT_DIR, exist_ok=True)

SAT_CSV_PATH = os.path.join(OUT_DIR, f"satellite{FLIGHT_ID}.csv") 
print(FLIGHT_CSV, UAV_IMG_PATH, SAT_IMG_PATH, SAT_CSV_PATH, sep="\n")

/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/03.csv
/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/drone/03_0001.JPG
/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/satellite03.tif
/home/nguyenduytan/UAV_nonGPS/UAV_nonGPS_dataset/03/output/satellite03.csv


In [ ]:
from UAV_nonGPS.matching.pipeline import LocalizationPipeline

# Dung chung mot cau hinh segmentation cho ca UAV va Satellite
COMMON_SEG = {
    "segmentation_dynamic_threshold": True,
    "segmentation_dynamic_quantile": 0.65,
    "segmentation_min_seg_threshold": 0.30,
    "segmentation_max_seg_threshold": 0.80,
    "segmentation_use_uncertainty_filter": True,
    "segmentation_uncertainty_ratio": 0.60,
}

pipe = LocalizationPipeline(model=model, device=device)
results = pipe.run(
    flight_csv=FLIGHT_CSV,
    uav_img_path=UAV_IMG_PATH,
    uav_csv_path_raw=FLIGHT_CSV,
    sat_img_path=SAT_IMG_PATH,
    sat_csv_path=SAT_CSV_PATH,
    enable_mask_crop_compare=True,

    compare_metric_mode="binary",
    compare_threshold="otsu",

    uav_soft_mask_scale=1.0,   # UAV compare mask dùng single-scale 1.0
    satellite_mask_scale=1.0,  # Satellite compare mask cũng 1.0

    uav_scales=[0.75, 1.0, 1.25, 1.5],
    uav_scale_weights=[0.15, 0.55, 0.20, 0.10],

    uav_patch_size=500,
    uav_overlap=100,
    satellite_patch_size=500,
    satellite_overlap=100,

    enable_local_offset_search=True,
    local_offset_max_px=120,
    local_offset_step_px=20,
    local_offset_metric="iou",

    **COMMON_SEG,
)


print(pipe.last_mask_compare_metrics["threshold_source"])
print(pipe.last_mask_compare_metrics["threshold"])


In [ ]:
# import importlib
# import UAV_nonGPS.matching.pipeline as pipeline_mod

# importlib.reload(pipeline_mod)
# LocalizationPipeline = pipeline_mod.LocalizationPipeline

# pipe = LocalizationPipeline(model=model, device=device)

# metrics = pipe.run_mask_crop_compare_only(
#     flight_csv=FLIGHT_CSV,
#     uav_img_path=UAV_IMG_PATH,
#     sat_img_path=SAT_IMG_PATH,
#     uav_csv_path_raw=FLIGHT_CSV,
#     output_dir=OUT_DIR,
#     satellite_mask_path=None,
#     compare_threshold="otsu",
#     satellite_mask_scale=1.0,
#     save_compare_figure=True,
#     use_local_satellite_crop=True,
#     satellite_crop_margin_scale=2.5,
#     satellite_patch_size=512,
#     satellite_overlap=200,
# )

# print(metrics["threshold_source"], metrics["threshold"])
# print(metrics["iou"], metrics["dice"], metrics["precision"], metrics["recall"])
# print(metrics["center_xy"], metrics.get("satellite_context_bbox_xyxy"))
